# 04 — Modelo Gold no grão de pedido

Objetivo: construir a estrela analítica. Entradas: três tabelas Silver e a regra única em sql/indicadores_entrega.sql. Saídas: dim_data, dim_localidade e fato_pedidos em Delta. Pré-requisitos: notebooks 01–03; executar a partir da Git folder que também contém o diretório sql. Evidências: contagens de junções, testes do grão e telas das tabelas/catálogo.

O grão da fato é uma linha por order_id. A soma de itens e frete ocorre antes da junção com pedidos. Nenhum outlier é descartado. Somas com item monetário faltante tornam-se nulas para evitar totais parciais apresentados como completos.


In [ ]:
from pyspark.sql import functions as F
import re

dbutils.widgets.text("catalogo", "")
catalogo = dbutils.widgets.get("catalogo").strip() or spark.sql("SELECT current_catalog()").first()[0]
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalogo), "Informe um catálogo válido no widget."
assert catalogo != "hive_metastore", "Use o catálogo do workspace com Unity Catalog."
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
origem = f"/Volumes/{catalogo}/olist_mvp_raw/originais"
silver = f"{catalogo}.olist_mvp_silver"
gold = f"{catalogo}.olist_mvp_gold"
arquivos = {
    "pedidos": "olist_orders_dataset.csv",
    "itens": "olist_order_items_dataset.csv",
    "clientes": "olist_customers_dataset.csv",
}
colunas_esperadas = {
    "pedidos": ["order_id", "customer_id", "order_status", "order_purchase_timestamp",
                "order_approved_at", "order_delivered_carrier_date",
                "order_delivered_customer_date", "order_estimated_delivery_date"],
    "itens": ["order_id", "order_item_id", "product_id", "seller_id",
              "shipping_limit_date", "price", "freight_value"],
    "clientes": ["customer_id", "customer_unique_id", "customer_zip_code_prefix",
                 "customer_city", "customer_state"],
}
print(f"Catálogo: {catalogo}; volume: {origem}")


In [ ]:
from pathlib import Path

pedidos = spark.table(f"{silver}.pedidos")
itens = spark.table(f"{silver}.itens")
clientes = spark.table(f"{silver}.clientes")
for nome, quadro, chave in [
    ("pedidos", pedidos, ["order_id"]), ("itens", itens, ["order_id", "order_item_id"]),
    ("clientes", clientes, ["customer_id"])]:
    assert quadro.groupBy(*chave).count().filter("count > 1").count() == 0, f"Duplicação em {nome}"


## Agregação dos itens antes de qualquer junção


In [ ]:
agregado = itens.groupBy("order_id").agg(
    F.count("*").alias("qtd_itens"),
    F.count("price").alias("qtd_precos_presentes"),
    F.count("freight_value").alias("qtd_fretes_presentes"),
    F.sum("price").alias("soma_precos"),
    F.sum("freight_value").alias("soma_fretes")
).select(
    "order_id", "qtd_itens",
    F.when(F.col("qtd_itens") == F.col("qtd_precos_presentes"), F.col("soma_precos")).alias("valor_itens_total"),
    F.when(F.col("qtd_itens") == F.col("qtd_fretes_presentes"), F.col("soma_fretes")).alias("frete_total")
)
assert agregado.count() == itens.select("order_id").distinct().count()
assert agregado.groupBy("order_id").count().filter("count > 1").count() == 0
display(agregado.agg(
    F.count("*").alias("pedidos_com_itens"),
    F.sum("qtd_itens").alias("itens_reconciliados"),
    F.sum(F.when(F.col("frete_total").isNull(), 1).otherwise(0)).alias("pedidos_frete_incompleto")
))
assert agregado.agg(F.sum("qtd_itens")).first()[0] == itens.count()


## Dimensões determinísticas


In [ ]:
local_base = clientes.select(
    "customer_id",
    F.col("customer_zip_code_prefix").alias("cep_prefixo"),
    F.col("customer_city").alias("cidade"),
    F.col("customer_state").alias("estado")
)
local_base = local_base.withColumn(
    "chave_localidade",
    F.sha2(F.to_json(F.struct("cep_prefixo", "cidade", "estado")), 256)
)
dim_localidade = local_base.select("chave_localidade", "cep_prefixo", "cidade", "estado").distinct()
assert dim_localidade.groupBy("chave_localidade").count().filter("count > 1").count() == 0

extremos = pedidos.agg(
    F.min(F.to_date("order_purchase_timestamp")).alias("inicio"),
    F.max(F.to_date("order_purchase_timestamp")).alias("fim")
).first()
assert extremos["inicio"] is not None and extremos["fim"] is not None, "Sem datas de compra"
assert (extremos["fim"] - extremos["inicio"]).days <= 4000, "Intervalo inesperado: investigar antes de gerar calendário"
dim_data = spark.sql(
    f"SELECT explode(sequence(date'{extremos['inicio']}', date'{extremos['fim']}', interval 1 day)) AS data_compra"
).select(
    "data_compra",
    F.year("data_compra").alias("ano"),
    F.month("data_compra").alias("mes"),
    F.date_format("data_compra", "yyyy-MM").alias("ano_mes")
)
assert dim_data.groupBy("data_compra").count().filter("count > 1").count() == 0
print(f"Calendário: {extremos['inicio']} a {extremos['fim']}; {dim_data.count()} dias")


## Indicadores comuns e fato


In [ ]:
raizes = [Path.cwd().parent / "sql", Path.cwd() / "sql"]
diretorio_sql = next((raiz for raiz in raizes if (raiz / "indicadores_entrega.sql").exists()), None)
assert diretorio_sql is not None, "Abra este notebook dentro da Git folder com o diretório sql."
regra = (diretorio_sql / "indicadores_entrega.sql").read_text(encoding="utf-8")
pedidos.createOrReplaceTempView("pedidos_silver_mvp")
pedidos_metricas = spark.sql(regra.format(pedidos="pedidos_silver_mvp"))
loc_cliente = local_base.select("customer_id", "chave_localidade")
assert loc_cliente.count() == clientes.count()

fato = (
    pedidos_metricas.alias("p")
    .join(loc_cliente.alias("c"), F.col("p.customer_id") == F.col("c.customer_id"), "left")
    .join(agregado.alias("i"), F.col("p.order_id") == F.col("i.order_id"), "left")
    .select(
        F.col("p.order_id").alias("order_id"),
        F.col("p.customer_id").alias("customer_id"),
        F.col("p.order_status").alias("order_status"),
        F.col("p.order_purchase_timestamp").alias("order_purchase_timestamp"),
        F.col("p.order_delivered_customer_date").alias("order_delivered_customer_date"),
        F.col("p.order_estimated_delivery_date").alias("order_estimated_delivery_date"),
        F.to_date(F.col("p.order_purchase_timestamp")).alias("data_compra"),
        F.col("c.chave_localidade").alias("chave_localidade"),
        F.col("i.qtd_itens").alias("qtd_itens"),
        F.col("i.valor_itens_total").alias("valor_itens_total"),
        F.col("i.frete_total").alias("frete_total"),
        F.col("p.pedido_entregue_com_datas").alias("pedido_entregue_com_datas"),
        F.col("p.entrega_atrasada").alias("entrega_atrasada")
    )
)


## Reconciliação e testes de integridade


In [ ]:
n_pedidos = pedidos.count()
n_fato = fato.count()
assert n_fato == n_pedidos, f"Junção multiplicou ou perdeu pedidos: {n_pedidos} → {n_fato}"
assert fato.filter(F.col("order_id").isNull()).count() == 0
assert fato.groupBy("order_id").count().filter("count > 1").count() == 0
assert fato.filter((F.col("pedido_entregue_com_datas") == 0) &
                   F.col("entrega_atrasada").isNotNull()).count() == 0
assert fato.filter((F.col("pedido_entregue_com_datas") == 1) &
                   F.col("entrega_atrasada").isNull()).count() == 0
assert fato.join(dim_localidade, "chave_localidade", "left_anti").filter(
    F.col("chave_localidade").isNotNull()).count() == 0
assert fato.join(dim_data, "data_compra", "left_anti").filter(
    F.col("data_compra").isNotNull()).count() == 0
print(f"Pedidos Silver: {n_pedidos}; fato: {n_fato}; diferença: {n_fato - n_pedidos}")
display(fato.agg(
    F.sum("pedido_entregue_com_datas").alias("pedidos_elegiveis"),
    F.sum("entrega_atrasada").alias("pedidos_atrasados"),
    F.sum(F.when(F.col("qtd_itens").isNull(), 1).otherwise(0)).alias("pedidos_sem_itens"),
    F.sum(F.when(F.col("chave_localidade").isNull(), 1).otherwise(0)).alias("pedidos_sem_cliente")
))


## Persistência Gold e comentários do catálogo


In [ ]:
saidas = {
    "dim_data": (dim_data, "Calendário da compra, um registro por dia.", {
        "data_compra": "Data civil da compra, chave da dimensão.",
        "ano": "Ano civil da compra.",
        "mes": "Mês civil da compra, de 1 a 12.",
        "ano_mes": "Ano e mês da compra no formato AAAA-MM.",
    }),
    "dim_localidade": (dim_localidade, "Localidade declarada do cliente.", {
        "chave_localidade": "Hash determinístico de CEP, cidade e UF.",
        "cep_prefixo": "Cinco dígitos iniciais do CEP do cliente.",
        "cidade": "Cidade normalizada em minúsculas.",
        "estado": "UF normalizada em maiúsculas.",
    }),
    "fato_pedidos": (fato, "Fato analítica com uma linha por pedido.", {
        "order_id": "Identificador único do pedido.",
        "customer_id": "Cliente ligado ao pedido.",
        "order_status": "Status do pedido normalizado.",
        "order_purchase_timestamp": "Instante original da compra.",
        "order_delivered_customer_date": "Instante real da entrega ao cliente.",
        "order_estimated_delivery_date": "Data prevista de entrega.",
        "data_compra": "Data civil derivada do instante da compra.",
        "chave_localidade": "Referência à dimensão de localidade do cliente.",
        "qtd_itens": "Quantidade de linhas de itens do pedido.",
        "valor_itens_total": "Soma completa dos preços dos itens em reais.",
        "frete_total": "Soma completa do frete dos itens em reais.",
        "pedido_entregue_com_datas": "1 se entregue com datas real e prevista; 0 caso contrário.",
        "entrega_atrasada": "1 se data real após prevista; 0 se pontual; nulo se inelegível.",
    }),
}
for nome, (quadro, descricao, colunas) in saidas.items():
    tabela = f"{gold}.{nome}"
    quadro.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    assert spark.table(tabela).count() == quadro.count()
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{descricao}'")
    for coluna, comentario in colunas.items():
        spark.sql(f"ALTER TABLE {tabela} ALTER COLUMN {coluna} COMMENT '{comentario}'")
    print(f"{tabela}: {quadro.count()} linhas persistidas")
